# Dead-End Discovery: Geometric Analysis of Discrete vs. Continuous Action Spaces

Compares three dead-end classifiers on **SpaceEnv** (10×10 map, planet at (5,5), GM=2, a_max=0.5):

1. **Ground truth (GT)**: circular boundary at $r_c = \sqrt{GM/a_{\max}} = 2.0$  
2. **Discrete grid (DeD)**: classify as dead-end if no grid action can overcome gravity. Uses $n_{\text{bins}} \times n_{\text{bins}}$ uniform grid over $[-a_{\max}, a_{\max}]^2$.  
3. **Uniform sampling (Cont-DeD)**: classify as dead-end if none of $N$ randomly sampled actions can overcome gravity.

**Key geometric insight.** The L$\infty$ action box provides maximum outward thrust  
$$a_{\text{max,radial}}(\varphi) = a_{\max}(|\cos\varphi| + |\sin\varphi|)$$  
in direction $\varphi$. This yields a *squircle* dead-end boundary  
$$r_c(\varphi) = \sqrt{\frac{GM}{a_{\max}(|\cos\varphi| + |\sin\varphi|)}}$$  
which ranges from $2.0$ (axis-aligned) to $\approx 1.682$ (diagonal). Since `np.linspace(-a_max, a_max, n_bins)` always includes $\pm a_{\max}$ as endpoints, **all discrete grids produce the same squircle boundary**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# ── SpaceEnv constants ──────────────────────────────────────────────────────
GM      = 2.0
A_MAX   = 0.5
PLANET  = np.array([5.0, 5.0])
MAP_SZ  = 10.0
R_BODY  = 1.0   # planet physical radius (from planet_specs=[[1,2]])
R_C     = np.sqrt(GM / A_MAX)   # = 2.0 — circular GT boundary

print(f"R_C (circular GT) = {R_C:.4f}")
print(f"Squircle min (diagonal) = {np.sqrt(GM / (A_MAX * np.sqrt(2))):.4f}")
print(f"Squircle max (axis)     = {np.sqrt(GM / A_MAX):.4f}")

In [ ]:
# ── Helper: squircle boundary ───────────────────────────────────────────────
def squircle_r(phi):
    """True L-inf dead-end radius at angle phi from planet."""
    return np.sqrt(GM / (A_MAX * (np.abs(np.cos(phi)) + np.abs(np.sin(phi)))))


# ── Helper: discrete action grid ────────────────────────────────────────────
def action_grid(n_bins):
    """n_bins×n_bins uniform grid over [-A_MAX, A_MAX]², shape (n_bins², 2)."""
    b = np.linspace(-A_MAX, A_MAX, n_bins)
    return np.array([(ax, ay) for ax in b for ay in b], dtype=np.float64)


# ── Helper: vectorised discrete classification ──────────────────────────────
def classify_discrete(positions, n_bins):
    """Return bool array (N,): True = classified as dead-end by discrete grid."""
    actions  = action_grid(n_bins)          # (n², 2)
    rel      = positions - PLANET           # (N, 2)
    r        = np.maximum(np.linalg.norm(rel, axis=1), 1e-8)  # (N,)
    outward  = rel / r[:, None]            # (N, 2) unit outward
    # max radial thrust available from the grid
    max_rad  = (outward @ actions.T).max(axis=1)  # (N,)
    gravity  = GM / r**2
    return max_rad < gravity               # can't overcome gravity


# ── Helper: vectorised sampling classification ──────────────────────────────
def classify_sampling(positions, n_sample_actions, seed=0):
    """Return bool array (N,): True = dead-end (no sampled action escapes)."""
    rng     = np.random.default_rng(seed)
    actions = rng.uniform(-A_MAX, A_MAX, size=(n_sample_actions, 2))
    rel     = positions - PLANET
    r       = np.maximum(np.linalg.norm(rel, axis=1), 1e-8)
    outward = rel / r[:, None]
    radial  = outward @ actions.T          # (N, n_sample_actions)
    gravity = GM / r**2
    any_esc = (radial > gravity[:, None]).any(axis=1)
    return ~any_esc


# ── Helper: analytical escape fraction ─────────────────────────────────────
def f_escape_numerical(positions, n_mc=50_000, seed=1):
    """
    Fraction of uniform actions that escape gravity at each position.
    Uses Monte-Carlo integration over the action box.
    Returns float array (N,) in [0, 1].
    """
    rng     = np.random.default_rng(seed)
    actions = rng.uniform(-A_MAX, A_MAX, size=(n_mc, 2))
    rel     = positions - PLANET
    r       = np.maximum(np.linalg.norm(rel, axis=1), 1e-8)
    outward = rel / r[:, None]
    radial  = outward @ actions.T          # (N, n_mc)
    gravity = GM / r**2
    return (radial > gravity[:, None]).mean(axis=1)

In [ ]:
# ── Build 2-D position grid ─────────────────────────────────────────────────
RES   = 250   # grid resolution
margin = 0.15
xs    = np.linspace(margin, MAP_SZ - margin, RES)
ys    = np.linspace(margin, MAP_SZ - margin, RES)
XX, YY = np.meshgrid(xs, ys)
pos   = np.stack([XX.ravel(), YY.ravel()], axis=1)  # (RES², 2)

rel_pos  = pos - PLANET
r_pos    = np.linalg.norm(rel_pos, axis=1)
phi_pos  = np.arctan2(rel_pos[:, 1], rel_pos[:, 0])

# Ground-truth dead-end (circular)
GT        = (r_pos < R_C).reshape(RES, RES)
# Planet body mask (not traversable)
body_mask = (r_pos < R_BODY).reshape(RES, RES)

print(f"GT dead-end pixels: {GT.sum()} / {RES**2}  ({100*GT.mean():.1f}%)")

In [ ]:
# ── Pre-compute discrete maps ───────────────────────────────────────────────
N_BINS_LIST  = [3, 5, 7, 9]
discrete_maps = {}
for nb in N_BINS_LIST:
    dm = classify_discrete(pos, nb).reshape(RES, RES)
    discrete_maps[nb] = dm
    n_actions = nb**2
    print(f"n_bins={nb}  ({n_actions:3d} actions): dead-end {100*dm.mean():.1f}%")

# Sanity: all discrete maps should be identical (squircle boundary)
maps = list(discrete_maps.values())
all_same = all((m == maps[0]).all() for m in maps[1:])
print(f"\nAll discrete maps identical: {all_same}  ← expected True")

In [ ]:
# ── Pre-compute sampling maps ───────────────────────────────────────────────
N_SAMPLES_LIST = [10, 32, 100, 1000]
sample_maps    = {}
for ns in N_SAMPLES_LIST:
    sm = classify_sampling(pos, ns, seed=42).reshape(RES, RES)
    sample_maps[ns] = sm
    print(f"n_samples={ns:5d}: dead-end {100*sm.mean():.1f}%")

In [ ]:
# ── Pre-compute Monte-Carlo escape fraction map (expensive, do once) ────────
print("Computing escape fraction map (50k MC samples)…", flush=True)
f_esc_flat = f_escape_numerical(pos, n_mc=50_000, seed=7)
F_ESC = f_esc_flat.reshape(RES, RES)
print("Done.")

## Figure 1 — Dead-End Boundary Maps

2 × 4 grid: top row = discrete grids (n=3,5,7,9), bottom row = sampling (N=10,32,100,1000).  
Red = classified dead-end; blue = classified safe; white dashed = GT r_c circle; grey = squircle boundary.

In [ ]:
phi_ring = np.linspace(0, 2 * np.pi, 500)
# GT circle
cx_gt = PLANET[0] + R_C * np.cos(phi_ring)
cy_gt = PLANET[1] + R_C * np.sin(phi_ring)
# Squircle
sr    = squircle_r(phi_ring)
cx_sq = PLANET[0] + sr * np.cos(phi_ring)
cy_sq = PLANET[1] + sr * np.sin(phi_ring)

# Colourmap: 0=safe (blue-white), 1=dead-end (red)
cmap_de = ListedColormap(['#cce5ff', '#ff7f7f'])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Dead-End Classification Maps', fontsize=15, fontweight='bold', y=1.01)

row_labels = [
    [f'Discrete  n={nb}×{nb}' for nb in N_BINS_LIST],
    [f'Sampling  N={ns}'       for ns in N_SAMPLES_LIST],
]
all_maps = [discrete_maps[nb] for nb in N_BINS_LIST] + \
           [sample_maps[ns]   for ns in N_SAMPLES_LIST]
all_labels = row_labels[0] + row_labels[1]

for idx, (ax, m, lbl) in enumerate(zip(axes.ravel(), all_maps, all_labels)):
    display = m.astype(float)
    display[body_mask] = np.nan     # hide planet body
    im = ax.imshow(display, origin='lower',
                   extent=[margin, MAP_SZ-margin, margin, MAP_SZ-margin],
                   cmap=cmap_de, vmin=0, vmax=1, interpolation='nearest')
    # GT circle
    ax.plot(cx_gt, cy_gt, 'w--', lw=1.4, label='GT $r_c$=2.0')
    # Squircle
    ax.plot(cx_sq, cy_sq, 'k:', lw=1.2, label='Squircle')
    # Planet body
    circ = plt.Circle(PLANET, R_BODY, color='#555555', zorder=5)
    ax.add_patch(circ)
    ax.set_title(lbl, fontsize=10)
    ax.set_xlim(margin, MAP_SZ-margin)
    ax.set_ylim(margin, MAP_SZ-margin)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    if idx == 0:
        ax.legend(loc='upper left', fontsize=7, framealpha=0.7)

# Row headers
for row, label in enumerate(['Discrete grid', 'Uniform sampling']):
    axes[row, 0].set_ylabel(label, fontsize=11, fontweight='bold')

# Legend patches
p_de  = mpatches.Patch(color='#ff7f7f', label='Classified dead-end')
p_safe= mpatches.Patch(color='#cce5ff', label='Classified safe')
fig.legend(handles=[p_de, p_safe], loc='lower center', ncol=2,
           bbox_to_anchor=(0.5, -0.04), fontsize=11)

plt.tight_layout()
plt.savefig('fig1_boundary_maps.pdf', bbox_inches='tight')
plt.show()

## Figure 2 — Angular Error Analysis

At three fixed radii (r = 1.75, 1.90, 2.05) we sweep angle φ ∈ [0°, 360°] and ask: does each classifier correctly label the state?

- **Discrete**: deterministic. Dead-end iff r < squircle_r(φ). Error at angles where squircle ≠ GT circle.
- **Sampling**: probabilistic. P(detect dead-end) = (1 − f_esc)^N, visualised for N=10,100,1000.

GT label at each (r, φ): dead-end iff r < R_C = 2.0.

In [ ]:
# Analytical escape fraction at given (r, phi) via MC
def f_esc_ring(r_val, phi_arr, n_mc=100_000, seed=3):
    """Returns f_esc for each phi in phi_arr at fixed radius r_val."""
    rng     = np.random.default_rng(seed)
    actions = rng.uniform(-A_MAX, A_MAX, size=(n_mc, 2))
    # positions on ring
    x = PLANET[0] + r_val * np.cos(phi_arr)
    y = PLANET[1] + r_val * np.sin(phi_arr)
    positions = np.stack([x, y], axis=1)
    rel     = positions - PLANET
    r       = np.maximum(np.linalg.norm(rel, axis=1), 1e-8)
    outward = rel / r[:, None]
    radial  = outward @ actions.T     # (n_phi, n_mc)
    gravity = (GM / r_val**2)
    return (radial > gravity).mean(axis=1)


r_vals   = [1.75, 1.90, 2.05]
phi_arr  = np.linspace(0, 2*np.pi, 720, endpoint=False)
phi_deg  = np.degrees(phi_arr)

N_vals   = [10, 100, 1000]
colors_N = ['#e6194b', '#f58231', '#3cb44b']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=False)
fig.suptitle('Angular Detection Probability at Fixed Radius', fontsize=13, fontweight='bold')

for ax, r_val in zip(axes, r_vals):
    gt_label = r_val < R_C   # all-or-nothing for a fixed r

    # --- Discrete: dead-end iff r < squircle_r(phi) ---
    sq_r = squircle_r(phi_arr)
    disc_de = (r_val < sq_r).astype(float)  # 1 = classified dead-end
    ax.fill_between(phi_deg, 0, disc_de, alpha=0.25, color='#4363d8',
                    label='Discrete (all n_bins)')
    ax.plot(phi_deg, disc_de, color='#4363d8', lw=1.5)

    # --- Sampling with N actions ---
    f_esc = f_esc_ring(r_val, phi_arr, n_mc=100_000, seed=3+int(r_val*10))
    for N_val, col in zip(N_vals, colors_N):
        p_de = (1 - f_esc) ** N_val   # P(all N actions fail) ≈ P(classify as dead-end)
        ax.plot(phi_deg, p_de, color=col, lw=1.8, label=f'Sampling N={N_val}')

    # GT horizontal line
    ax.axhline(1.0 if gt_label else 0.0, color='k', lw=2, ls='--', label='GT label')

    ax.set_title(f'r = {r_val}  ({"dead-end" if gt_label else "safe"} by GT)', fontsize=11)
    ax.set_xlabel('Angle from planet (°)')
    ax.set_ylabel('P(classified as dead-end)')
    ax.set_xlim(0, 360)
    ax.set_ylim(-0.05, 1.12)
    ax.set_xticks([0, 90, 180, 270, 360])
    ax.grid(True, alpha=0.3)

    # Mark squircle minima (diagonal angles: 45°, 135°, 225°, 315°)
    for dg in [45, 135, 225, 315]:
        ax.axvline(dg, color='grey', lw=0.7, ls=':')

axes[0].legend(fontsize=8, loc='lower right', framealpha=0.8)

plt.tight_layout()
plt.savefig('fig2_angular_error.pdf', bbox_inches='tight')
plt.show()

## Figure 3 — Classification Error Area vs. Computational Cost

- **Discrete FN area**: fraction of GT dead-end pixels incorrectly labelled safe (annular squircle–circle region near diagonals).
- **Discrete FP area**: fraction of GT safe pixels incorrectly labelled dead-end (zero — discrete always includes corners).
- **Sampling FP area**: fraction of GT safe pixels labelled dead-end (states just outside r_c that look dead-end by chance).
- **Sampling FN area**: fraction of GT dead-end pixels that sampling misses.

x-axis = number of action evaluations per state (n_bins² for discrete, N for sampling).

In [ ]:
def error_areas(pred_map, gt_map, body_mask):
    """Return (FP_frac, FN_frac) relative to all non-body pixels."""
    valid = ~body_mask
    n_valid = valid.sum()
    FP = float((pred_map & ~gt_map & valid).sum()) / n_valid
    FN = float((~pred_map & gt_map & valid).sum()) / n_valid
    return FP, FN


# Discrete
disc_costs, disc_FP, disc_FN = [], [], []
for nb in N_BINS_LIST:
    fp, fn = error_areas(discrete_maps[nb], GT, body_mask)
    disc_costs.append(nb**2)
    disc_FP.append(fp)
    disc_FN.append(fn)

# Sampling
samp_costs, samp_FP, samp_FN = [], [], []
for ns in N_SAMPLES_LIST:
    fp, fn = error_areas(sample_maps[ns], GT, body_mask)
    samp_costs.append(ns)
    samp_FP.append(fp)
    samp_FN.append(fn)

print("Discrete (FP always 0 since corners included):")
for nb, fp, fn in zip(N_BINS_LIST, disc_FP, disc_FN):
    print(f"  n={nb}  FP={fp:.4f}  FN={fn:.4f}")

print("\nSampling:")
for ns, fp, fn in zip(N_SAMPLES_LIST, samp_FP, samp_FN):
    print(f"  N={ns:5d}  FP={fp:.4f}  FN={fn:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Classification Error Area vs. Computational Cost', fontsize=13, fontweight='bold')

# --- Left: False Negatives ---
ax = axes[0]
ax.plot(disc_costs, disc_FN, 'o--', color='#4363d8', lw=2, ms=8, label='Discrete (squircle–circle annulus)')
ax.plot(samp_costs, samp_FN, 's-', color='#f58231', lw=2, ms=8, label='Uniform sampling')
ax.set_xscale('log')
ax.set_xlabel('Number of actions evaluated per state')
ax.set_ylabel('FN fraction (missed dead-ends / total non-body pixels)')
ax.set_title('False Negatives (missed dead-ends)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.35)

# Annotate discrete plateau
ax.annotate('Discrete FN = const. (squircle\ncovers all diagonal escapes)',
            xy=(disc_costs[1], disc_FN[1]),
            xytext=(30, disc_FN[1]+0.01),
            arrowprops=dict(arrowstyle='->', color='#4363d8'),
            fontsize=8, color='#4363d8')

# --- Right: False Positives ---
ax = axes[1]
ax.plot(disc_costs, disc_FP, 'o--', color='#4363d8', lw=2, ms=8, label='Discrete')
ax.plot(samp_costs, samp_FP, 's-', color='#f58231', lw=2, ms=8, label='Uniform sampling')
ax.set_xscale('log')
ax.set_xlabel('Number of actions evaluated per state')
ax.set_ylabel('FP fraction (safe states misclassified / total non-body pixels)')
ax.set_title('False Positives (false alarms)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig('fig3_error_area_vs_cost.pdf', bbox_inches='tight')
plt.show()

## Figure 4 — Boundary Precision vs. Computational Cost (IoU and F1 in boundary band)

Restrict evaluation to the boundary band $|r - r_c| < \delta$ (δ = 0.5) where errors concentrate.
Compute IoU and F1 between the classifier's map and the GT map within this band.

In [ ]:
DELTA_BAND = 0.5
band_mask  = (np.abs(r_pos - R_C) < DELTA_BAND).reshape(RES, RES) & ~body_mask

print(f"Band pixels: {band_mask.sum()}  ({100*band_mask.mean():.1f}% of map)")


def boundary_metrics(pred_map, gt_map, band):
    """IoU and F1 of pred vs gt restricted to band."""
    p  = pred_map[band]
    g  = gt_map[band]
    TP = (p & g).sum()
    FP = (p & ~g).sum()
    FN = (~p & g).sum()
    iou = TP / max(TP + FP + FN, 1)
    f1  = 2*TP / max(2*TP + FP + FN, 1)
    return float(iou), float(f1)


disc_iou, disc_f1 = [], []
for nb in N_BINS_LIST:
    iou, f1 = boundary_metrics(discrete_maps[nb], GT, band_mask)
    disc_iou.append(iou); disc_f1.append(f1)
    print(f"Discrete n={nb}: IoU={iou:.3f}  F1={f1:.3f}")

print()
samp_iou, samp_f1 = [], []
for ns in N_SAMPLES_LIST:
    iou, f1 = boundary_metrics(sample_maps[ns], GT, band_mask)
    samp_iou.append(iou); samp_f1.append(f1)
    print(f"Sampling N={ns:5d}: IoU={iou:.3f}  F1={f1:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Boundary Precision vs. Computational Cost  (band |r − r_c| < {DELTA_BAND})',
             fontsize=13, fontweight='bold')

for ax, (disc_metric, samp_metric, ylabel, title) in zip(
    axes,
    [(disc_iou, samp_iou, 'IoU', 'Intersection-over-Union'),
     (disc_f1,  samp_f1,  'F1',  'F1 Score')]
):
    ax.plot(disc_costs, disc_metric, 'o--', color='#4363d8', lw=2, ms=9, label='Discrete grid')
    ax.plot(samp_costs, samp_metric, 's-',  color='#f58231', lw=2, ms=9, label='Uniform sampling')

    # Ideal reference line
    all_costs = sorted(disc_costs + samp_costs)
    ax.axhline(1.0, color='k', lw=1, ls=':', alpha=0.5, label='Perfect (1.0)')

    ax.set_xscale('log')
    ax.set_xlabel('Number of actions evaluated per state')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_ylim(0, 1.08)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.35)

    # Annotate discrete plateau
    ax.annotate('Discrete: constant\n(squircle boundary)',
                xy=(disc_costs[2], disc_metric[2]),
                xytext=(disc_costs[2]*1.3, disc_metric[2]-0.12),
                arrowprops=dict(arrowstyle='->', color='#4363d8'),
                fontsize=8, color='#4363d8')

plt.tight_layout()
plt.savefig('fig4_boundary_precision_vs_cost.pdf', bbox_inches='tight')
plt.show()

## Figure 5 — Escape Cone Coverage vs. Distance from Planet

At each radius r, what fraction of safe states are *correctly* identified as safe (recall on safe states, i.e. TNR = 1 − FPR)?  
And what fraction of dead-end states are correctly identified as dead-end (TPR = recall on dead-ends)?

We also overlay the **analytical escape fraction** $f_{\text{esc}}(r)$ averaged over all angles — the fundamental limit on sampling performance.

In [ ]:
# Build radial bins
r_bins   = np.linspace(R_BODY + 0.05, 5.0, 60)
r_centres= 0.5 * (r_bins[:-1] + r_bins[1:])
dr       = r_bins[1] - r_bins[0]

r_flat   = r_pos                      # (RES²,)
gt_flat  = GT.ravel()                 # (RES²,)
body_flat= body_mask.ravel()          # (RES²,)


def radial_tpr_tnr(pred_flat, r_flat, gt_flat, body_flat, r_centres, dr):
    """TPR and TNR (dead-end recall, safe recall) vs radius."""
    tpr = np.full(len(r_centres), np.nan)
    tnr = np.full(len(r_centres), np.nan)
    for i, rc in enumerate(r_centres):
        mask = (~body_flat) & (np.abs(r_flat - rc) < dr/2)
        if mask.sum() == 0:
            continue
        p = pred_flat[mask]; g = gt_flat[mask]
        de_mask = g; safe_mask = ~g
        if de_mask.sum() > 0:
            tpr[i] = p[de_mask].mean()      # TP / (TP+FN)
        if safe_mask.sum() > 0:
            tnr[i] = (~p)[safe_mask].mean() # TN / (TN+FP)
    return tpr, tnr


# Analytical f_esc per radial bin (average over angles in that bin)
f_esc_radial = np.full(len(r_centres), np.nan)
f_esc_flat_arr = F_ESC.ravel()
for i, rc in enumerate(r_centres):
    mask = (~body_flat) & (np.abs(r_flat - rc) < dr/2)
    if mask.sum() > 0:
        f_esc_radial[i] = f_esc_flat_arr[mask].mean()

print("Radial analysis computed.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Escape Cone Coverage vs. Distance from Planet', fontsize=13, fontweight='bold')

# Choose one discrete grid (n=5) and two sampling sizes for clarity
configs = [
    ('Discrete n=5', discrete_maps[5].ravel(), '#4363d8', '--'),
    ('Sampling N=32',  sample_maps[32].ravel(),  '#e6194b', '-'),
    ('Sampling N=100', sample_maps[100].ravel(), '#f58231', '-'),
    ('Sampling N=1000',sample_maps[1000].ravel(),'#3cb44b', '-'),
]

# --- Left: TPR (recall on dead-ends) ---
ax = axes[0]
for label, pred, col, ls in configs:
    tpr, _ = radial_tpr_tnr(pred, r_flat, gt_flat, body_flat, r_centres, dr)
    ax.plot(r_centres, tpr, color=col, ls=ls, lw=2, label=label)

# Analytical TPR for sampling: P(all N fail | state is dead-end by GT)
# For states inside squircle: f_esc=0 → sampling always finds dead-end → TPR=1
# For states in [squircle, r_c]: f_esc>0 → P(TPR) = (1-f_esc)^N
# We show the expected analytical TPR for N=100
p_de_100 = np.where(np.isnan(f_esc_radial), np.nan, (1 - f_esc_radial)**100)
# Only meaningful inside GT circle
gt_in_radial = r_centres < R_C
ax.plot(r_centres[gt_in_radial], p_de_100[gt_in_radial], 'k:', lw=1.5,
        label='Analytical E[TPR] N=100')

ax.axvline(R_C,  color='grey', lw=1.5, ls='--', label=f'GT r_c={R_C}')
ax.axvline(np.sqrt(GM/(A_MAX*np.sqrt(2))), color='silver', lw=1, ls=':',
           label='Squircle min')
ax.set_xlabel('Distance from planet r  (map units)')
ax.set_ylabel('TPR — dead-end recall')
ax.set_title('True Positive Rate on Dead-End States')
ax.set_xlim(R_BODY, 5.0)
ax.set_ylim(-0.05, 1.12)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.35)

# --- Right: TNR (recall on safe states) ---
ax = axes[1]
for label, pred, col, ls in configs:
    _, tnr = radial_tpr_tnr(pred, r_flat, gt_flat, body_flat, r_centres, dr)
    ax.plot(r_centres, tnr, color=col, ls=ls, lw=2, label=label)

# Analytical TNR for sampling: for safe states (r > R_C), TNR = 1-(1-f_esc)^N
p_safe_100 = np.where(np.isnan(f_esc_radial), np.nan, 1 - (1 - f_esc_radial)**100)
gt_out_radial = r_centres >= R_C
ax.plot(r_centres[gt_out_radial], p_safe_100[gt_out_radial], 'k:', lw=1.5,
        label='Analytical E[TNR] N=100')

ax.axvline(R_C,  color='grey', lw=1.5, ls='--', label=f'GT r_c={R_C}')
ax.set_xlabel('Distance from planet r  (map units)')
ax.set_ylabel('TNR — safe state recall')
ax.set_title('True Negative Rate on Safe States')
ax.set_xlim(R_BODY, 5.0)
ax.set_ylim(-0.05, 1.12)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig('fig5_escape_coverage_vs_distance.pdf', bbox_inches='tight')
plt.show()

## Summary Table

Global FP, FN, IoU, F1 across the full map (excluding planet body).

In [ ]:
def all_metrics(pred, gt, body):
    valid = ~body
    p = pred[valid]; g = gt[valid]
    TP = (p & g).sum();  FP = (p & ~g).sum()
    FN = (~p & g).sum(); TN = (~p & ~g).sum()
    n  = valid.sum()
    iou = TP / max(TP+FP+FN, 1)
    f1  = 2*TP / max(2*TP+FP+FN, 1)
    acc = (TP+TN) / n
    return dict(FP_pct=100*FP/n, FN_pct=100*FN/n,
                IoU=round(iou,3), F1=round(f1,3), Acc=round(acc,3))

print(f"{'Config':<22} {'FP%':>6} {'FN%':>6} {'IoU':>7} {'F1':>7} {'Acc':>7}")
print('-'*60)
for nb in N_BINS_LIST:
    m = all_metrics(discrete_maps[nb], GT, body_mask)
    print(f"Discrete  n={nb}×{nb}      {m['FP_pct']:6.2f} {m['FN_pct']:6.2f} {m['IoU']:7.3f} {m['F1']:7.3f} {m['Acc']:7.3f}")
print()
for ns in N_SAMPLES_LIST:
    m = all_metrics(sample_maps[ns], GT, body_mask)
    print(f"Sampling  N={ns:<7}      {m['FP_pct']:6.2f} {m['FN_pct']:6.2f} {m['IoU']:7.3f} {m['F1']:7.3f} {m['Acc']:7.3f}")

## Key Findings

### Discrete grid
- **All n_bins grids produce identical results.** Since `np.linspace(-a_max, a_max, n_bins)` always includes $\pm a_{\max}$ as endpoints, the corner actions `(±a_{\max}, ±a_{\max})` are present for every grid size. The maximum achievable radial thrust is $a_{\max}(|\cos\varphi|+|\sin\varphi|)$ regardless of $n_{\text{bins}}$.
- **Boundary = squircle, not the GT circle.** The discrete dead-end boundary follows $r_c(\varphi) = \sqrt{GM\,/\,(a_{\max}(|\cos\varphi|+|\sin\varphi|))}$, ranging from 2.0 (axis) to ≈1.682 (diagonal). This is more conservative than the GT circle at axis directions and more lenient at diagonals.
- **False Negatives near diagonals.** States with $r_{\text{squircle}}(\varphi) < r < r_c$ are labelled *safe* by the discrete grid (the corner action provides enough diagonal thrust), but are dead-ends by the GT circular criterion.
- **Zero False Positives.** No safe state (r > r_c) is ever mis-classified as dead-end.

### Uniform sampling
- **False Positives decrease with N.** Near $r_c$ from outside, $f_{\text{esc}}$ is small, so with few samples none escape and the state is labelled dead-end. Increasing N rapidly drives FP → 0.
- **False Negatives inside squircle = 0.** Inside the squircle, $f_{\text{esc}}=0$ by definition, so all actions fail and the state is always correctly labelled dead-end.
- **FN in the annular band [squircle, r_c]** arise because $f_{\text{esc}} > 0$; the probability of all N samples missing the escape cone is $(1-f_{\text{esc}})^N$, which decays exponentially with N.
- **Boundary sharpens with N** and converges to the squircle (same as discrete) for $N \to \infty$.

### Comparative
| Metric | Discrete (any n_bins) | Sampling N=1000 |
|--------|----------------------|------------------|
| FP | **0** | Near 0 |
| FN | Fixed (squircle–circle annulus) | Near 0 |
| Boundary shape | Squircle | Converges to squircle |
| Computational cost | n_bins² (≤ 81) | N (1000+) |

The discrete grid is highly efficient but introduces a systematic, geometry-dependent FN region. Sampling reduces this FN at the cost of more action evaluations, and additionally introduces FP errors at small N that disappear as N grows.